In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google API 키가 존재하며 {google_api_key[:8]}로 시작합니다")
else:
    print("Google API 키가 설정되어 있지 않습니다")

Google API 키가 존재하며 AQ.Ab8RN로 시작합니다


In [11]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.1-flash-lite"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [4]:
system_message = "당신은 도움이 되는 어시스턴트입니다"


In [5]:
# 콜백 함수 작성
def chat(message, history):
    return "bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [7]:
def chat(message, history):
    return f"당신은 {message}라고 말했고 history는 {history}이지만, 그래도 저는 bananas라고 하겠습니다"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [9]:
# 좋아요! 좀 더 나은 chat 콜백을 작성해봅시다!

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [13]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = gemini.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


# 시스템 메시지를 사용해 맥락을 추가하고 예시 답변을 제공합니다.. 이것 역시 "원샷 프롬프팅"입니다

In [15]:
system_message = "당신은 옷가게에서 일하는 도움이 되는 어시스턴트입니다. \
세일 중인 상품을 고객이 사보도록 부드럽게 유도해야 합니다. 모자는 60% 할인 중이고, 대부분의 다른 상품은 50% 할인 중입니다. \
예를 들어 고객이 '모자를 사고 싶어요'라고 말하면, \
'좋습니다 - 저희는 세일 이벤트에 포함된 것을 포함해 다양한 모자를 보유하고 있어요'와 같이 답할 수 있습니다.\
고객이 무엇을 살지 확신하지 못할 때는 모자 구매를 권유하세요."

In [16]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


In [17]:
system_message += "\n고객이 신발을 찾으면, 오늘은 신발이 세일 중이 아니라고 답하되, \
모자를 살펴보라고 다시 한번 권유하세요!"

In [18]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


In [20]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if '벨트' in message.lower():
        relevant_system_message += " 이 가게는 벨트를 판매하지 않습니다; 벨트를 요청받으면, 세일 중인 다른 상품을 반드시 안내하세요."

    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = gemini.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [21]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.
